In [1]:
import os
import pandas as pd
directory = "dataset/relaion2B-en-research-safe"
parquet_file = "0001.parquet"
df = pd.read_parquet(os.path.join(directory, parquet_file))
print(df.head())

                                                 url  similarity  \
0  https://lid.zoocdn.com/354/255/1c262cde0e91356...    0.331089   
1  http://tse2.mm.bing.net/th?id=OIP.Gqs8tMg9Ncak...    0.357405   
2  https://shop.foleyfoodandwinesociety.com/asset...    0.312547   
3  https://d1ea30dbll17d8.cloudfront.net/12/1/ima...    0.317522   
4  http://rlv.zcache.com/pomeranian_christmas_car...    0.353620   

                  hash  pwatermark   punsafe  \
0  9011316736760728558    0.022310  0.000020   
1  9174018384016294622    0.051443  0.000004   
2 -1427967606297722939    0.152133  0.001647   
3  -601349193925773650    0.027884  0.000724   
4 -3973424436504122004    0.004785  0.000010   

                                             caption         key   status  \
0  5 bed detached house for sale in The Middlings...  1519780945  success   
1  Sink Vanity Cabinet Bathroom Design 72 Quot Ba...  2192073997  success   
2                        Acrobat Rosé (12x375ml can)  0142410570  succe

In [ ]:
# The hash & md5 although meant to identify duplicates, don't seem to do so as all urls having the same md5 were different and none of the hashes overlapped

# hash_counts = df['md5'].value_counts()
urls_with_md5 = df[df['md5'] == '7e38a78e5dc8f67ae17b6eb76a25348c']['url'].tolist()
print(urls_with_md5)
print("Total URLs with specified md5:", len(urls_with_md5))
print("Distinct URLs with specified md5:", len(set(urls_with_md5)))

['https://i.ebayimg.com/images/g/~TUAAOSw3xJVW1hm/s-l225.jpg', 'https://i.ebayimg.com/thumbs/images/g/jMsAAOSww9xZNzeY/s-l225.jpg', 'https://www.picclickimg.com/d/l400/pict/264115550031_/Ciele-Athletics-Go-Cap-Athletics-Pop-Running-Fitness.jpg', 'https://i.ebayimg.com/images/i/262128820965-0-1/s-l1000.jpg', 'https://thumbs.ebaystatic.com/images/g/PM8AAOSwo4pYKydF/s-l225.jpg', 'https://i.ebayimg.com/thumbs/images/g/jfsAAOSwunJf1RUp/s-l300.jpg', 'https://i.ebayimg.com/images/g/nDMAAOSwYFpbRcq8/s-l1600.jpg', 'https://www.picclickimg.com/d/l400/pict/222624771349_/PUNK-ROCK-METAL-MUSIC-FESTIVAL-RUBBER-WRISTBAND-BRACELET-NIRVANA.jpg', 'https://i.ebayimg.com/thumbs/images/g/iiAAAOSwTelbmM~k/s-l225.jpg', 'https://www.picclickimg.com/d/l400/pict/303160595622_/Handmade-PLUSH-fleece-tie-blanket-of-rainy.jpg', 'https://i.ebayimg.com/images/g/sfYAAOSw701f7hf3/s-l400.jpg', 'https://thumbs2.ebaystatic.com/d/l225/m/muSKAOgXjCiiX3j8Fpye7UA.jpg', 'https://i.ebayimg.com/thumbs/images/g/gr4AAOSw8PpfU0Bz/s

In [ ]:
# None of the top pwatermark URLs appear to actually have watermarks
url_pwatermark_sorted = df[['url', 'pwatermark']].sort_values('pwatermark', ascending=False)
print(url_pwatermark_sorted.head(10))
index = 2585345   
print(url_pwatermark_sorted['url'][index], url_pwatermark_sorted['pwatermark'][index])

                                                        url  pwatermark
14443360  https://tse3.mm.bing.net/th?id=OIP.bSdwt51hC1f...         1.0
9036177   https://cdn.shopify.com/s/files/1/0103/8526/06...         1.0
2585225   https://images.bonanzastatic.com/afu/images/38...         1.0
14387744  http://t0.gstatic.com/images?q=tbn:ANd9GcQu4i2...         1.0
14387742  https://seasondistribution.de/media/image/prod...         1.0
6748569   https://img1.etsystatic.com/218/1/13151120/il_...         1.0
8713134   http://l.yimg.com/bt/api/res/1.2/xLXGCHNWAWfUf...         1.0
11164601  https://cdn4.vectorstock.com/i/thumb-large/30/...         1.0
15604921  https://image.tmdb.org/t/p/w1280/js3J4SBiRfLvm...         1.0
2585345   https://ecdn.teacherspayteachers.com/thumbitem...         1.0
https://ecdn.teacherspayteachers.com/thumbitem/Foodborne-Illneses-Divide-and-Conquer--2453639/large-2453639-1.jpg 1.0


#### Test 1 - Testing Execution Time Using Data Loader

In [ ]:
import os
import time
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm

# 1. Define a simple transform (resize + convert to tensor)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# 2. Custom Dataset for flat folder
class FlatFolderDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.files = [os.path.join(root_dir, f)
                      for f in os.listdir(root_dir)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_path = self.files[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img

# 3. Path to your images in 0000/images
image_folder_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\clip_embeddings_resumable_symlink\downloaded_images\0000"

# 4. Dataset & DataLoader
dataset = FlatFolderDataset(image_folder_dir, transform=transform)
loader = DataLoader(dataset,
                    batch_size=256,  # adjust depending on RAM
                    shuffle=False,
                    num_workers=0,   # parallel workers
                    pin_memory=True)

# 5. Dummy watermark function
def dummy_detect_watermark(batch):
    return (batch.mean(dim=[1, 2, 3]) > 0.5).numpy()

# 6. Process all images with progress bar
start = time.time()
count = 0

for batch in tqdm(loader, total=len(loader), desc="Processing images in 0000"):
    results = dummy_detect_watermark(batch)
    count += len(results)

elapsed = time.time() - start
print(f"\n✅ Processed {count} images in {elapsed:.2f} seconds")
print(f"⚡ Throughput: {count/elapsed:.2f} images/sec")


Processing images in 0000:  15%|█▌        | 49/320 [02:07<11:45,  2.60s/it]

In [ ]:
import os
import time
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

class FlatFolderDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.files = [os.path.join(root_dir, f)
                      for f in os.listdir(root_dir)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_path = self.files[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img

image_folder_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\clip_embeddings_resumable_symlink\downloaded_images\0000"
dataset = FlatFolderDataset(image_folder_dir, transform=transform)

loader = DataLoader(dataset,
                    batch_size=256,
                    shuffle=False,
                    num_workers=0,       # multi-worker
                    pin_memory=True)#,
                    # prefetch_factor=4)

def dummy_detect_watermark(batch):
    return (batch.mean(dim=[1, 2, 3]) > 0.5).numpy()

start = time.time()
count = 0
pbar = tqdm(total=len(loader), desc="Processing images in 0000")

for batch in loader:
    results = dummy_detect_watermark(batch)
    count += len(results)
    pbar.update(1)

pbar.close()

elapsed = time.time() - start
print(f"\n✅ Processed {count} images in {elapsed:.2f} seconds")
print(f"⚡ Throughput: {count/elapsed:.2f} images/sec")


Processing images in 0000:  29%|██▉       | 94/320 [02:44<11:12,  2.98s/it]

In [ ]:
import os
import time
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm

# 1️⃣ Define a simple transform (resize + convert to tensor)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# 2️⃣ Custom Dataset for a flat folder
class FlatFolderDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.files = [os.path.join(root_dir, f)
                      for f in os.listdir(root_dir)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_path = self.files[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img

# 3️⃣ Path to your images in 0000/images
image_folder_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\clip_embeddings_resumable_symlink\downloaded_images\0000"

# 4️⃣ Dataset & DataLoader
dataset = FlatFolderDataset(image_folder_dir, transform=transform)
loader = DataLoader(dataset,
                    batch_size=1,        # adjust depending on RAM
                    shuffle=False,
                    num_workers=8,         # multi-worker for parallel loading
                    pin_memory=True,
                    prefetch_factor=4)     # preload more batches per worker

# 5️⃣ Dummy watermark function
def dummy_detect_watermark(batch):
    # Example: check average brightness
    return (batch.mean(dim=[1, 2, 3]) > 0.5).numpy()

# 6️⃣ Process all images with manual tqdm update
start = time.time()
count = 0
pbar = tqdm(total=len(loader), desc="Processing images in 0000")

for batch in loader:
    results = dummy_detect_watermark(batch)
    count += len(results)
    pbar.update(1)  # manually increment progress bar per batch

pbar.close()

elapsed = time.time() - start
print(f"\n✅ Processed {count} images in {elapsed:.2f} seconds")
print(f"⚡ Throughput: {count/elapsed:.2f} images/sec")


Processing images in 0000:   0%|          | 0/81802 [00:00<?, ?it/s]

In [2]:
import os
import time
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm

class FlatFolderDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.files = [os.path.join(root_dir, f)
                      for f in os.listdir(root_dir)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_path = self.files[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img

def dummy_detect_watermark(batch):
    return (batch.mean(dim=[1, 2, 3]) > 0.5).numpy()


if __name__ == "__main__":
    # Transform
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])

    # Path to images
    image_folder_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\clip_embeddings_resumable_symlink\downloaded_images\0000"

    # Dataset & DataLoader
    dataset = FlatFolderDataset(image_folder_dir, transform=transform)
    # The key change: We wrap the DataLoader directly with tqdm
    loader = DataLoader(dataset,
                        batch_size=1,
                        shuffle=False,
                        num_workers=0,
                        pin_memory=True)#,
                        # prefetch_factor=4)

    start = time.time()
    count = 0

    # Wrap the loader with tqdm directly for an elegant solution
    for batch in tqdm(loader, desc="Processing images in 0000"):
        results = dummy_detect_watermark(batch)
        count += len(results)

    elapsed = time.time() - start
    print(f"\n✅ Processed {count} images in {elapsed:.2f} seconds")
    print(f"⚡ Throughput: {count/elapsed:.2f} images/sec")


Processing images in 0000: 100%|██████████| 81802/81802 [12:06<00:00, 112.59it/s]


✅ Processed 81802 images in 726.54 seconds
⚡ Throughput: 112.59 images/sec


#### Test 2. IQA with dummy watermark function (takes too long with IQA)

In [ ]:
import os
import time
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
# from tqdm import tqdm
from tqdm.notebook import tqdm
import pyiqa

# --- transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# --- Dataset
class FlatFolderDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.files = [os.path.join(root_dir, f)
                      for f in os.listdir(root_dir)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_path = self.files[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, os.path.basename(img_path)

# --- Path
image_folder_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\clip_embeddings_resumable_symlink\downloaded_images\0000"
dataset = FlatFolderDataset(image_folder_dir, transform=transform)

torch.multiprocessing.set_start_method('spawn', force=True)

loader = DataLoader(dataset,
                    batch_size=8,         # IQA models are heavy; smaller batches
                    shuffle=False,
                    num_workers=0,
                    pin_memory=True)

# --- Device
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- Load IQA models
models = {
    'paq2piq': pyiqa.create_metric('paq2piq').to(device)#,
    # 'brisque': pyiqa.create_metric('brisque').to(device),
    # 'maniqa': pyiqa.create_metric('maniqa-pipal').to(device),
    # 'niqe': pyiqa.create_metric('niqe').to(device)
}

# --- Dummy watermark function
def dummy_detect_watermark(batch):
    return (batch.mean(dim=[1, 2, 3]) > 0.5).cpu().numpy()

# --- Storage for results
results = []

# --- Loop over batches
pbar = tqdm(loader, desc="Processing images", total=len(loader))
for batch, filenames in pbar:
    batch = batch.to(device)

    # --- Watermark detection
    wm_results = dummy_detect_watermark(batch)

    # --- Compute IQA scores per model
    iqa_scores = {}
    for name, model in models.items():
        with torch.no_grad():
            scores = model(batch).detach().cpu().numpy()
        iqa_scores[name] = scores

    # --- Combine results per image
    for i, fname in enumerate(filenames):
        results.append({
            'filename': fname,
            'watermark': bool(wm_results[i]),
            'paq2piq': float(iqa_scores['paq2piq'][i])#,
            # 'brisque': float(iqa_scores['brisque'][i]),
            # 'maniqa': float(iqa_scores['maniqa'][i]),
            # 'niqe': float(iqa_scores['niqe'][i])
        })

pbar.close()

# --- Convert to DataFrame for convenience
import pandas as pd
df = pd.DataFrame(results)
print(df.head())


c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.conda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading pretrained model PAQ2PIQ from C:\Users\User\.cache\torch\hub\pyiqa\P2P_RoIPoolModel-fit.10.bs.120-ca69882e.pth


Processing images:   0%|          | 0/10226 [00:00<?, ?it/s]C:\Users\User\AppData\Local\Temp\ipykernel_15056\1291787167.py:86: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  'paq2piq': float(iqa_scores['paq2piq'][i])#,
Processing images:   1%|▏         | 128/10226 [04:16<130:26:29, 46.50s/it]

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.notebook import tqdm
import pyiqa


# --- Dataset definition
class FlatFolderDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.files = [
            os.path.join(root_dir, f)
            for f in os.listdir(root_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))
        ]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_path = self.files[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, os.path.basename(img_path)


# --- Dummy watermark function
def dummy_detect_watermark(batch):
    return (batch.mean(dim=[1, 2, 3]) > 0.5).cpu().numpy()


# --- Main function
def main():
    # --- transforms
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])

    # --- Path
    image_folder_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\clip_embeddings_resumable_symlink\downloaded_images\0000"
    dataset = FlatFolderDataset(image_folder_dir, transform=transform)

    # --- DataLoader
    loader = DataLoader(
        dataset,
        batch_size=8,
        shuffle=False,
        num_workers=4,   # <-- can increase when running as .py
        pin_memory=True
    )

    # --- Device
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # --- Load IQA models
    models = {
        'paq2piq': pyiqa.create_metric('paq2piq').to(device)
    }

    # --- Storage for results
    results = []

    # --- Loop over batches
    pbar = tqdm(loader, desc="Processing images", total=len(loader))
    for batch, filenames in pbar:
        batch = batch.to(device)

        # --- Watermark detection
        wm_results = dummy_detect_watermark(batch)

        # --- Compute IQA scores per model
        iqa_scores = {}
        for name, model in models.items():
            with torch.no_grad():
                scores = model(batch).detach().cpu().numpy()
            iqa_scores[name] = scores

        # --- Combine results per image
        for i, fname in enumerate(filenames):
            results.append({
                'filename': fname,
                'watermark': bool(wm_results[i]),
                'paq2piq': float(iqa_scores['paq2piq'][i])
            })

    pbar.close()

    # --- Convert to DataFrame for convenience
    df = pd.DataFrame(results)
    print(df.head())


# --- Entry point (safe for multiprocessing)
if __name__ == "__main__":
    torch.multiprocessing.set_start_method('spawn', force=True)
    main()


#### Test 3 - OWLv2 Testing (Uses region level CLIP embeddings unlike my global scale CLIP embeddings)

In [1]:
import requests
from PIL import Image
import torch

from transformers import Owlv2Processor, Owlv2ForObjectDetection

processor = Owlv2Processor.from_pretrained("google/owlv2-large-patch14-ensemble")
model = Owlv2ForObjectDetection.from_pretrained("google/owlv2-large-patch14-ensemble")

print("Model and processor loaded.")

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)
texts = [["a photo of a cat"]]#, "a photo of a dog"]]
inputs = processor(text=texts, images=image, return_tensors="pt")

with torch.no_grad():
  outputs = model(**inputs)

# Target image sizes (height, width) to rescale box predictions [batch_size, 2]
target_sizes = torch.Tensor([image.size[::-1]])
# Convert outputs (bounding boxes and class logits) to Pascal VOC Format (xmin, ymin, xmax, ymax)
results = processor.post_process_object_detection(outputs=outputs, target_sizes=target_sizes, threshold=0.1)
i = 0  # Retrieve predictions for the first image for the corresponding text queries
text = texts[i]
boxes, scores, labels = results[i]["boxes"], results[i]["scores"], results[i]["labels"]
for box, score, label in zip(boxes, scores, labels):
    box = [round(i, 2) for i in box.tolist()]
    print(f"Detected {text[label]} with confidence {round(score.item(), 3)} at location {box}")


c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.conda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.conda\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model and processor loaded.
Detected a photo of a cat with confidence 0.596 at location [340.21, 18.56, 639.03, 277.13]
Detected a photo of a cat with confidence 0.578 at location [11.53, 41.62, 315.83, 353.76]


In [1]:
import requests
from PIL import Image
import torch
from transformers import Owlv2Processor, Owlv2ForObjectDetection
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load processor and model
processor = Owlv2Processor.from_pretrained("google/owlv2-large-patch14-ensemble")
model = Owlv2ForObjectDetection.from_pretrained("google/owlv2-large-patch14-ensemble").to(device)
print("Model and processor loaded.")

# Load image
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)
texts = [["a photo of a cat"]]

# Prepare inputs and move to GPU
inputs = processor(text=texts, images=image, return_tensors="pt").to(device)

# Run inference on GPU
with torch.no_grad():
    outputs = model(**inputs)  # Keep as model output object

# Target image sizes (height, width)
target_sizes = torch.Tensor([image.size[::-1]])  # image.size = (width, height)

# Post-process predictions directly using the output object
results = processor.post_process_object_detection(
    outputs=outputs,  # <-- pass the original output object
    target_sizes=target_sizes,
    threshold=0.1
)

# Retrieve predictions for the first image
i = 0
text_labels = texts[i]
boxes, scores, labels = results[i]["boxes"], results[i]["scores"], results[i]["labels"]

for box, score, label in zip(boxes, scores, labels):
    box = [round(v, 2) for v in box.tolist()]
    print(f"Detected {text_labels[label]} with confidence {round(score.item(), 3)} at location {box}")

c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.conda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.conda\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model and processor loaded.
Detected a photo of a cat with confidence 0.596 at location [340.21, 18.56, 639.03, 277.13]
Detected a photo of a cat with confidence 0.578 at location [11.53, 41.62, 315.83, 353.76]


In [12]:
import requests
from PIL import Image
import torch
from transformers import Owlv2Processor, Owlv2ForObjectDetection
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import time  # Import time module

# Use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load processor and model
processor = Owlv2Processor.from_pretrained("google/owlv2-large-patch14-ensemble")
model = Owlv2ForObjectDetection.from_pretrained("google/owlv2-large-patch14-ensemble").to(device)
print("Model and processor loaded.")

# # Load image
# url = "http://images.cocodataset.org/val2017/000000039769.jpg"
# image = Image.open(requests.get(url, stream=True).raw)

image = Image.open(r'C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\clip_embeddings_resumable_symlink\downloaded_images\0000\65.jpg')
texts = [["a photo of a cat", "watermark", "logo"]]

# Prepare inputs and move to GPU
inputs = processor(text=texts, images=image, return_tensors="pt").to(device)

# Start timer
start_time = time.time()

# Run inference on GPU
with torch.no_grad():
    outputs = model(**inputs)  # Keep as model output object

# Target image sizes (height, width)
target_sizes = torch.Tensor([image.size[::-1]])  # image.size = (width, height)

# Post-process predictions directly using the output object
results = processor.post_process_object_detection(
    outputs=outputs,  # <-- pass the original output object
    target_sizes=target_sizes,
    threshold=0.1
)

# End timer
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Processing time: {elapsed_time:.3f} seconds")

# Retrieve predictions for the first image
i = 0
text_labels = texts[i]
boxes, scores, labels = results[i]["boxes"], results[i]["scores"], results[i]["labels"]

for box, score, label in zip(boxes, scores, labels):
    box = [round(v, 2) for v in box.tolist()]
    print(f"Detected {text_labels[label]} with confidence {round(score.item(), 3)} at location {box}")


Using device: cuda
Model and processor loaded.
Processing time: 1.187 seconds
Detected logo with confidence 0.157 at location [-0.93, 0.76, 301.02, 131.12]
Detected logo with confidence 0.393 at location [10.0, 42.02, 289.6, 83.74]
Detected logo with confidence 0.153 at location [10.13, 40.75, 74.36, 80.92]
Detected watermark with confidence 0.123 at location [70.4, 53.56, 284.44, 84.69]


In [7]:
import requests
from PIL import Image, ImageDraw, ImageFont
import torch
from transformers import Owlv2Processor, Owlv2ForObjectDetection
import time

# Use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load processor and model
processor = Owlv2Processor.from_pretrained("google/owlv2-large-patch14-ensemble")
model = Owlv2ForObjectDetection.from_pretrained("google/owlv2-large-patch14-ensemble").to(device)
print("Model and processor loaded.")

# Load image
# image_path = 'C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/downloaded_images/0000/65.jpg'
image_path = 'C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/downloaded_images/0000/3.jpg'
image = Image.open(image_path).convert("RGB")
texts = [["a watermark"]]

# Prepare inputs and move to GPU
inputs = processor(text=texts, images=image, return_tensors="pt").to(device)

# Start timer
start_time = time.time()

# Run inference on GPU
with torch.no_grad():
    outputs = model(**inputs)

# Target image sizes (height, width)
target_sizes = torch.Tensor([image.size[::-1]])

# Post-process predictions
results = processor.post_process_object_detection(
    outputs=outputs,
    target_sizes=target_sizes,
    threshold=0.1
)

# End timer
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Processing time: {elapsed_time:.3f} seconds")

# Retrieve predictions for the first image
i = 0
text_labels = texts[i]
boxes, scores, labels = results[i]["boxes"], results[i]["scores"], results[i]["labels"]

# Create a drawing object
draw = ImageDraw.Draw(image)

try:
    # Try to use a default font
    font = ImageFont.truetype("arial.ttf", 15)
except IOError:
    # Fallback to a basic font if 'arial.ttf' isn't found
    font = ImageFont.load_default()

for box, score, label in zip(boxes, scores, labels):
    box = [round(v, 2) for v in box.tolist()]
    x_min, y_min, x_max, y_max = box
    label_text = f"{text_labels[label]}: {score.item():.2f}"
    
    # Draw the rectangle
    draw.rectangle([x_min, y_min, x_max, y_max], outline="red", width=3)
    
    # Use textbbox() to get the bounding box of the text
    left, top, right, bottom = draw.textbbox((x_min, y_min), label_text, font=font)
    text_width = right - left
    text_height = bottom - top

    # Draw the text background
    draw.rectangle([x_min, y_min - text_height, x_min + text_width, y_min], fill="red")
    draw.text((x_min, y_min - text_height), label_text, fill="white", font=font)

# Save or display the image with bounding boxes
image.show() 
print("Image with bounding boxes saved as 'image_with_boxes.jpg'")

Using device: cuda
Model and processor loaded.
Processing time: 1.256 seconds
Image with bounding boxes saved as 'image_with_boxes.jpg'


In [ ]:
import os
from PIL import Image
from ultralytics import YOLO
import torchvision.transforms.functional as TVF
from transformers import Owlv2VisionModel
from torch import nn
import torch
import torch.nn.functional as F
import requests
import io
import time

# OWLv2 classification head
class DetectorModelOwl(nn.Module):
    owl: Owlv2VisionModel

    def __init__(self, model_path: str, dropout: float, n_hidden: int = 768):
        super().__init__()

        owl = Owlv2VisionModel.from_pretrained(model_path)
        assert isinstance(owl, Owlv2VisionModel)
        self.owl = owl
        self.owl.requires_grad_(False)
        self.transforms = None

        self.dropout1 = nn.Dropout(dropout)
        self.ln1 = nn.LayerNorm(n_hidden, eps=1e-5)
        self.linear1 = nn.Linear(n_hidden, n_hidden * 2)
        self.act1 = nn.GELU()
        self.dropout2 = nn.Dropout(dropout)
        self.ln2 = nn.LayerNorm(n_hidden * 2, eps=1e-5)
        self.linear2 = nn.Linear(n_hidden * 2, 2)
    
    def forward(self, pixel_values: torch.Tensor, labels: torch.Tensor | None = None):
        with torch.autocast("cpu", dtype=torch.bfloat16):
            # Embed the image
            outputs = self.owl(pixel_values=pixel_values, output_hidden_states=True)
            x = outputs.last_hidden_state  # B, N, C
        
            # Linear
            x = self.dropout1(x)
            x = self.ln1(x)
            x = self.linear1(x)
            x = self.act1(x)

            # Norm and Mean
            x = self.dropout2(x)
            x, _ = x.max(dim=1)
            x = self.ln2(x)

            # Linear
            x = self.linear2(x)
        
        if labels is not None:
            loss = F.cross_entropy(x, labels)
            return (x, loss)

        return (x,)


def owl_predict(image: Image.Image) -> bool:
    # Pad to square
    big_side = max(image.size)
    new_image = Image.new("RGB", (big_side, big_side), (128, 128, 128))
    new_image.paste(image, (0, 0))

    # Resize to 960x960
    preped = new_image.resize((960, 960), Image.BICUBIC)

    # Convert to tensor and normalize
    preped = TVF.pil_to_tensor(preped)
    preped = preped / 255.0
    input_image = TVF.normalize(preped, [0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711])

    # Run model
    logits, = owl_model(input_image.to('cpu').unsqueeze(0), None)
    probs = F.softmax(logits, dim=1)
    prediction = torch.argmax(probs.cpu(), dim=1)

    return prediction.item() == 1


# def yolo_predict(image: Image.Image):
#     results = yolo_model(image, imgsz=1024, augment=True, iou=0.5)
#     assert len(results) == 1
#     result = results[0]
#     im_array = result.plot()
#     return Image.fromarray(im_array[..., ::-1])


def process_image(image_path: str):
    try:
        image = Image.open(image_path)
    except FileNotFoundError:
        print(f"Error: Image not found at {image_path}")
        return
    except Exception as e:
        print(f"Error loading image at {image_path}: {e}")
        return

    # OWLv2 prediction
    owl_prediction = owl_predict(image)
    label_owl = "Watermarked" if owl_prediction else "Not Watermarked"
    
    # # YOLO prediction
    # yolo_image = yolo_predict(image)

    # # Display or save the results
    print(f"OWLv2 Prediction: {label_owl}")
    # yolo_image.show(title=f"YOLO Detections for {os.path.basename(image_path)}")
    
    # You can save the image instead of showing it
    # yolo_image.save(f"detected_{os.path.basename(image_path)}")


# --- Main execution block ---
if __name__ == "__main__":
    # Load models
    try:
        owl_model = DetectorModelOwl("google/owlv2-base-patch16-ensemble", dropout=0.0)
        owl_model.load_state_dict(torch.load("far5y1y5-8000.pt", map_location="cpu"))
        owl_model.eval()
        
        # yolo_model = YOLO("yolo11x-train28-best.pt")
        print("Models loaded successfully.")
    except Exception as e:
        print(f"Error loading models: {e}")
        exit()

    # Define the image path you want to process
    # Replace this with the actual path to your image
    my_image_path = 'C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/downloaded_images/0000/3.jpg'

    # Process the single image
    process_image(my_image_path)

C:\Users\User\AppData\Local\Temp\ipykernel_28780\3146385137.py:121: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  owl_model.load_state_dict(torch.load("far5y1y5-8000.pt", ma

Models loaded successfully.


In [9]:
import os
from PIL import Image
from ultralytics import YOLO
import torchvision.transforms.functional as TVF
from transformers import Owlv2VisionModel
from torch import nn
import torch
import torch.nn.functional as F
import time
import cv2

# --- General Setup ---
# Check for GPU availability and define the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# OWLv2 classification head
class DetectorModelOwl(nn.Module):
    owl: Owlv2VisionModel

    def __init__(self, model_path: str, dropout: float, n_hidden: int = 768):
        super().__init__()

        # Move the base model to the GPU
        owl = Owlv2VisionModel.from_pretrained(model_path).to(device)
        assert isinstance(owl, Owlv2VisionModel)
        self.owl = owl
        self.owl.requires_grad_(False)
        self.transforms = None

        # The rest of your model's layers
        self.dropout1 = nn.Dropout(dropout)
        self.ln1 = nn.LayerNorm(n_hidden, eps=1e-5)
        self.linear1 = nn.Linear(n_hidden, n_hidden * 2)
        self.act1 = nn.GELU()
        self.dropout2 = nn.Dropout(dropout)
        self.ln2 = nn.LayerNorm(n_hidden * 2, eps=1e-5)
        self.linear2 = nn.Linear(n_hidden * 2, 2)
    
    def forward(self, pixel_values: torch.Tensor, labels: torch.Tensor | None = None):
        # Use autocast for mixed precision on GPU to save memory and speed up
        # Switch to "cuda"
        with torch.autocast(device.type, dtype=torch.bfloat16):
            # Embed the image
            outputs = self.owl(pixel_values=pixel_values, output_hidden_states=True)
            x = outputs.last_hidden_state  # B, N, C
        
            # Linear
            x = self.dropout1(x)
            x = self.ln1(x)
            x = self.linear1(x)
            x = self.act1(x)

            # Norm and Mean
            x = self.dropout2(x)
            x, _ = x.max(dim=1)
            x = self.ln2(x)

            # Linear
            x = self.linear2(x)
        
        if labels is not None:
            # Move labels to device
            labels = labels.to(device)
            loss = F.cross_entropy(x, labels)
            return (x, loss)

        return (x,)


def owl_predict(image: Image.Image) -> bool:
    # Pad to square
    big_side = max(image.size)
    new_image = Image.new("RGB", (big_side, big_side), (128, 128, 128))
    new_image.paste(image, (0, 0))

    # Resize to 960x960
    preped = new_image.resize((960, 960), Image.BICUBIC)

    # Convert to tensor and normalize
    preped = TVF.pil_to_tensor(preped)
    preped = preped / 255.0
    input_image = TVF.normalize(preped, [0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711])

    # Move the input tensor to the GPU before running the model
    input_image = input_image.to(device).unsqueeze(0)

    # Run model
    with torch.no_grad(): # Use no_grad for inference
        logits, = owl_model(input_image, None)
    
    # Move results back to CPU for numpy/PIL operations
    probs = F.softmax(logits, dim=1)
    prediction = torch.argmax(probs.cpu(), dim=1)
    print(f"OWLv2 Prediction logits: {logits}, probs: {probs}, prediction: {prediction}")

    return prediction.item() == 1


# The YOLO part is commented out, but if you want to use it, uncomment and ensure the yolo_model is also moved to the device.
def yolo_predict(image: Image.Image):
    results = yolo_model(image, imgsz=1024, augment=True, iou=0.5)
    assert len(results) == 1
    result = results[0]
    im_array = result.plot()
    # The array is returned from the GPU, move it back to CPU to convert to PIL image
    return Image.fromarray(im_array[..., ::-1])


def process_image(image_path: str):
    try:
        image = Image.open(image_path)
    except FileNotFoundError:
        print(f"Error: Image not found at {image_path}")
        return
    except Exception as e:
        print(f"Error loading image at {image_path}: {e}")
        return

    start_time = time.time()
    
    # OWLv2 prediction
    owl_prediction = owl_predict(image)
    label_owl = "Watermarked" if owl_prediction else "Not Watermarked"
    
    # YOLO prediction
    yolo_image = yolo_predict(image)

    end_time = time.time()
    print(f"Processing time: {end_time - start_time:.2f} seconds")

    print(f"OWLv2 Prediction: {label_owl}")
    yolo_image.show(title=f"YOLO Detections for {os.path.basename(image_path)}")


# --- Main execution block ---
if __name__ == "__main__":
    try:
        # Load the OWLv2 model and move it to the GPU
        owl_model = DetectorModelOwl("google/owlv2-base-patch16-ensemble", dropout=0.0).to(device)
        owl_model.load_state_dict(torch.load("far5y1y5-8000.pt", map_location=device))
        owl_model.eval()
        
        # If using YOLO, move it to the device as well
        yolo_model = YOLO("yolo11x-train28-best.pt").to(device)
        print("Models loaded successfully and moved to GPU.")
    except Exception as e:
        print(f"Error loading models or moving to GPU: {e}")
        exit()

    # Define the image path
    my_image_path = 'C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/downloaded_images/0000/3.jpg'

    # Process the single image
    process_image(my_image_path)

Using device: cuda


c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.conda\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\User\AppData\Local\Temp\ipykernel_30948\730330529.py:141: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allow

Models loaded successfully and moved to GPU.
OWLv2 Prediction logits: tensor([[-2.4219,  2.4062]], device='cuda:0', dtype=torch.bfloat16), probs: tensor([[0.0079, 0.9922]], device='cuda:0', dtype=torch.bfloat16), prediction: tensor([1])

0: 768x1024 34 watermarks, 115.4ms
Speed: 11.7ms preprocess, 115.4ms inference, 1.3ms postprocess per image at shape (1, 3, 768, 1024)
Processing time: 0.66 seconds
OWLv2 Prediction: Watermarked


In [12]:
import os
from PIL import Image
from ultralytics import YOLO
import torchvision.transforms.functional as TVF
from transformers import Owlv2VisionModel
from torch import nn
import torch
import torch.nn.functional as F
import time
import cv2

# --- General Setup ---
# Check for GPU availability and define the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# OWLv2 classification head
class DetectorModelOwl(nn.Module):
    owl: Owlv2VisionModel

    def __init__(self, model_path: str, dropout: float, n_hidden: int = 768):
        super().__init__()
        owl = Owlv2VisionModel.from_pretrained(model_path).to(device)
        assert isinstance(owl, Owlv2VisionModel)
        self.owl = owl
        self.owl.requires_grad_(False)
        self.transforms = None

        self.dropout1 = nn.Dropout(dropout)
        self.ln1 = nn.LayerNorm(n_hidden, eps=1e-5)
        self.linear1 = nn.Linear(n_hidden, n_hidden * 2)
        self.act1 = nn.GELU()
        self.dropout2 = nn.Dropout(dropout)
        self.ln2 = nn.LayerNorm(n_hidden * 2, eps=1e-5)
        self.linear2 = nn.Linear(n_hidden * 2, 2)
    
    def forward(self, pixel_values: torch.Tensor, labels: torch.Tensor | None = None):
        with torch.autocast(device.type, dtype=torch.bfloat16):
            outputs = self.owl(pixel_values=pixel_values, output_hidden_states=True)
            x = outputs.last_hidden_state
            x = self.dropout1(x)
            x = self.ln1(x)
            x = self.linear1(x)
            x = self.act1(x)
            x = self.dropout2(x)
            x, _ = x.max(dim=1)
            x = self.ln2(x)
            x = self.linear2(x)
        
        if labels is not None:
            labels = labels.to(device)
            loss = F.cross_entropy(x, labels)
            return (x, loss)

        return (x,)


# Define the models as global variables to be loaded only once
owl_model = None
yolo_model = None

def load_models():
    """Loads and returns the pre-trained OWLv2 and YOLO models."""
    global owl_model, yolo_model
    if owl_model is None:
        try:
            owl_model = DetectorModelOwl("google/owlv2-base-patch16-ensemble", dropout=0.0).to(device)
            owl_model.load_state_dict(torch.load("far5y1y5-8000.pt", map_location=device))
            owl_model.eval()
            print("OWLv2 model loaded.")
        except Exception as e:
            print(f"Error loading OWLv2 model: {e}")
            return False

    if yolo_model is None:
        try:
            yolo_model = YOLO("yolo11x-train28-best.pt").to(device)
            print("YOLO model loaded.")
        except Exception as e:
            print(f"Error loading YOLO model: {e}")
            return False
            
    return True


def watermark_detector(
    image_path: str, 
    run_yolo: bool, 
    yolo_conf_thresh: float, 
    owl_conf_thresh: float
):
    """
    Detects watermarks and bounding boxes in an image using OWLv2 and YOLO models.

    Args:
        image_path (str): The path to the image file.
        run_yolo (bool): Flag to enable/disable YOLO bounding box detection.
        yolo_conf_thresh (float): Confidence threshold for YOLO detections (0.0 to 1.0).
        owl_conf_thresh (float): Confidence threshold for OWLv2 watermarking classification (0.0 to 1.0).

    Returns:
        dict: A dictionary containing the image path, watermark status, and YOLO bounding boxes.
              Returns None if the image cannot be processed.
    """
    
    # Check if models are loaded; if not, load them
    if not (owl_model and yolo_model):
        if not load_models():
            return None

    try:
        image = Image.open(image_path).convert("RGB")
    except FileNotFoundError:
        print(f"Error: Image not found at {image_path}")
        return None
    except Exception as e:
        print(f"Error loading image at {image_path}: {e}")
        return None

    start_time = time.time()
    
    # OWLv2 Prediction
    # Pad image to a square shape
    big_side = max(image.size)
    new_image = Image.new("RGB", (big_side, big_side), (128, 128, 128))
    new_image.paste(image, (0, 0))

    # Resize and normalize for OWLv2 model
    preped = new_image.resize((960, 960), Image.BICUBIC)
    preped = TVF.pil_to_tensor(preped) / 255.0
    input_image = TVF.normalize(preped, [0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711])
    input_image = input_image.to(device).unsqueeze(0)
    
    with torch.no_grad():
        logits, = owl_model(input_image, None)
    
    probs = F.softmax(logits, dim=1)
    watermark_prob = probs[0][1].item()
    is_watermarked = watermark_prob >= owl_conf_thresh
    
    # YOLO Prediction
    yolo_boxes = []
    if run_yolo:
        results = yolo_model(image, imgsz=1024, augment=True, iou=0.5, conf=yolo_conf_thresh)
        if results:
            result = results[0]
            for box in result.boxes:
                coords = box.xyxy[0].tolist()
                yolo_boxes.append({
                    "class_id": int(box.cls.item()),
                    "confidence": float(box.conf.item()),
                    "bbox_coords": [round(c, 2) for c in coords]
                })

    end_time = time.time()
    total_time = end_time - start_time
    print(f"Total processing time for {os.path.basename(image_path)}: {total_time:.2f} seconds")

    return {
        "image_path": image_path,
        "is_watermarked": is_watermarked,
        "yolo_boxes": yolo_boxes
    }

# --- Main execution block ---
if __name__ == "__main__":
    # Define the image path
    my_image_path = 'C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/downloaded_images/0000/3.jpg'
    
    # Example usage of the new function
    results = watermark_detector(
        image_path=my_image_path,
        run_yolo=True,  # Set to False to skip YOLO detection
        yolo_conf_thresh=0.7,
        owl_conf_thresh=0.8
    )

    if results:
        print("\n--- Final Results ---")
        print(f"Image Path: {results['image_path']}")
        print(f"Is Watermarked: {results['is_watermarked']}")
        print(f"YOLO Bounding Boxes: {results['yolo_boxes']}")

Using device: cuda


C:\Users\User\AppData\Local\Temp\ipykernel_30948\2620360101.py:68: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  owl_model.load_state_dict(torch.load("far5y1y5-8000.pt", map

OWLv2 model loaded.
YOLO model loaded.

0: 768x1024 32 watermarks, 118.5ms
Speed: 4.0ms preprocess, 118.5ms inference, 1.2ms postprocess per image at shape (1, 3, 768, 1024)
Total processing time for 3.jpg: 0.74 seconds

--- Final Results ---
Image Path: C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/downloaded_images/0000/3.jpg
Is Watermarked: True
YOLO Bounding Boxes: [{'class_id': 0, 'confidence': 0.955069899559021, 'bbox_coords': [152.56, 272.44, 261.88, 327.96]}, {'class_id': 0, 'confidence': 0.9548203945159912, 'bbox_coords': [1042.68, 273.37, 1151.79, 328.72]}, {'class_id': 0, 'confidence': 0.9546473026275635, 'bbox_coords': [889.03, 663.38, 1000.03, 719.29]}, {'class_id': 0, 'confidence': 0.9510390162467957, 'bbox_coords': [595.42, 402.48, 704.17, 458.58]}, {'class_id': 0, 'confidence': 0.9499261379241943, 'bbox_coords': [0.0, 662.61, 103.91, 719.16]}, {'class_id': 0, 'confidence': 0.9132263660430908, 'bbox_coords': [1240.42, 15

In [2]:
import os
from PIL import Image
from ultralytics import YOLO
import torchvision.transforms.functional as TVF
from transformers import Owlv2VisionModel
from torch import nn
import torch
import torch.nn.functional as F
import time
import cv2
from concurrent.futures import ThreadPoolExecutor

# --- General Setup ---
# Check for GPU availability and define the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# OWLv2 classification head
class DetectorModelOwl(nn.Module):
    owl: Owlv2VisionModel

    def __init__(self, model_path: str, dropout: float, n_hidden: int = 768):
        super().__init__()

        owl = Owlv2VisionModel.from_pretrained(model_path).to(device)
        assert isinstance(owl, Owlv2VisionModel)
        self.owl = owl
        self.owl.requires_grad_(False)
        self.transforms = None

        self.dropout1 = nn.Dropout(dropout)
        self.ln1 = nn.LayerNorm(n_hidden, eps=1e-5)
        self.linear1 = nn.Linear(n_hidden, n_hidden * 2)
        self.act1 = nn.GELU()
        self.dropout2 = nn.Dropout(dropout)
        self.ln2 = nn.LayerNorm(n_hidden * 2, eps=1e-5)
        self.linear2 = nn.Linear(n_hidden * 2, 2)
    
    def forward(self, pixel_values: torch.Tensor, labels: torch.Tensor | None = None):
        with torch.autocast(device.type, dtype=torch.bfloat16):
            # Embed the image
            outputs = self.owl(pixel_values=pixel_values, output_hidden_states=True)
            x = outputs.last_hidden_state # B, N, C

            # Linear
            x = self.dropout1(x)
            x = self.ln1(x)
            x = self.linear1(x)
            x = self.act1(x)

            # Norm and Mean
            x = self.dropout2(x)
            x, _ = x.max(dim=1)
            x = self.ln2(x)

            # Linear
            x = self.linear2(x)
        
        if labels is not None:
            labels = labels.to(device)
            loss = F.cross_entropy(x, labels)
            return (x, loss)

        return (x,)


# Define the models as global variables to be loaded only once
owl_model = None
yolo_model = None

def load_models():
    """Loads and returns the pre-trained OWLv2 and YOLO models."""
    global owl_model, yolo_model
    if owl_model is None:
        try:
            owl_model = DetectorModelOwl("google/owlv2-base-patch16-ensemble", dropout=0.0).to(device)
            owl_model.load_state_dict(torch.load("far5y1y5-8000.pt", map_location=device))
            owl_model.eval()
            print("OWLv2 model loaded.")
        except Exception as e:
            print(f"Error loading OWLv2 model: {e}")
            return False

    if yolo_model is None:
        try:
            yolo_model = YOLO("yolo11x-train28-best.pt").to(device)
            print("YOLO model loaded.")
        except Exception as e:
            print(f"Error loading YOLO model: {e}")
            return False
            
    return True

def preprocess_image_for_owl(image_path: str) -> torch.Tensor:
    """Preprocesses a PIL image for OWLv2 model input."""
    try:
        image = Image.open(image_path).convert("RGB")

        # OWLv2 preprocessing (pad, resize, normalize)
        big_side = max(image.size)
        new_image = Image.new("RGB", (big_side, big_side), (128, 128, 128))
        new_image.paste(image, (0, 0))
        preped = new_image.resize((960, 960), Image.BICUBIC)

        preped = TVF.pil_to_tensor(preped) / 255.0
        input_image = TVF.normalize(preped, [0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711])
        return input_image
    except FileNotFoundError:
        print(f"Error: Image not found at {image_path}. Skipping.")
        return None
    except Exception as e:
        print(f"Error loading image at {image_path}: {e}. Skipping.")
        return None


def watermark_detector_batch(
    image_paths: list[str], 
    run_yolo: bool, 
    yolo_conf_thresh: float, 
    owl_conf_thresh: float
):
    """
    Detects watermarks and bounding boxes in a batch of images using OWLv2 and YOLO models.

    Args:
        image_paths (list[str]): A list of paths to the image files.
        run_yolo (bool): Flag to enable/disable YOLO bounding box detection.
        yolo_conf_thresh (float): Confidence threshold for YOLO detections (0.0 to 1.0).
        owl_conf_thresh (float): Confidence threshold for OWLv2 watermarking classification (0.0 to 1.0).

    Returns:
        list[dict]: A list of dictionaries, each containing the results for one image.
                    Returns None if the models cannot be loaded.
    """
    # Check if models are loaded; if not, load them
    if not (owl_model and yolo_model):
        if not load_models():
            return None

    results = []
    
    # --- Prepare Batch for OWLv2 Prediction ---
    start_time = time.time()
    
    owl_inputs = []
    yolo_image_list = []

    # # Preprocessing the batched images in parallel
    # with ThreadPoolExecutor(max_workers=os.cpu_count()) as executor:
    #     owl_inputs = list(executor.map(preprocess_image_for_owl, image_paths))

    for image_path in image_paths:
        try:
            image = Image.open(image_path).convert("RGB")
            
            # OWLv2 preprocessing (pad, resize, normalize)
            big_side = max(image.size)
            new_image = Image.new("RGB", (big_side, big_side), (128, 128, 128))
            new_image.paste(image, (0, 0))
            preped = new_image.resize((960, 960), Image.BICUBIC)
            
            preped = TVF.pil_to_tensor(preped) / 255.0
            input_image = TVF.normalize(preped, [0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711])
            owl_inputs.append(input_image)
            
            # yolo_image_list.append(image)
            
        except FileNotFoundError:
            print(f"Error: Image not found at {image_path}. Skipping.")
            results.append({"image_path": image_path, "is_watermarked": False, "yolo_boxes": []})
        except Exception as e:
            print(f"Error loading image at {image_path}: {e}. Skipping.")
            results.append({"image_path": image_path, "is_watermarked": False, "yolo_boxes": []})

    if not owl_inputs:
        return results

    # Concatenate all preprocessed images into a single batch tensor
    owl_batch = torch.stack(owl_inputs).to(device)

    # OWLv2 Batched Prediction
    with torch.no_grad():
        logits, = owl_model(owl_batch, None)
    
    probs = F.softmax(logits, dim=1)
    
    for i, image_path in enumerate(image_paths):
        watermark_prob = probs[i][1].item()
        is_watermarked = watermark_prob >= owl_conf_thresh
        
        # Initialize results for the current image
        results.append({
            "image_path": image_path,
            "is_watermarked": is_watermarked,
            "yolo_boxes": []
        })

    # --- YOLO Batched Prediction ---
    if run_yolo:
        # Pass the list of PIL images directly to YOLO for batched processing
        yolo_results = yolo_model(yolo_image_list, imgsz=1024, augment=True, iou=0.5, conf=yolo_conf_thresh)
        
        # Process YOLO results and add them to the output dictionaries
        for i, result in enumerate(yolo_results):
            yolo_boxes = []
            for box in result.boxes:
                coords = box.xyxy[0].tolist()
                yolo_boxes.append({
                    "class_id": int(box.cls.item()),
                    "confidence": float(box.conf.item()),
                    "bbox_coords": [round(c, 2) for c in coords]
                })
            results[i]["yolo_boxes"] = yolo_boxes

    end_time = time.time()
    total_time = end_time - start_time
    print(f"Total processing time for batch of {len(image_paths)} images: {total_time:.2f} seconds")

    return results

# --- Main execution block ---
if __name__ == "__main__":
    # Define the directory and the processing limit
    image_directory = 'C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/downloaded_images/0000'
    process_limit = 10  # Set the maximum number of images to process

    # Get a list of all files in the directory
    all_files = os.listdir(image_directory)

    # Filter for common image file extensions and create full paths
    image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp']
    all_image_paths = [
        os.path.join(image_directory, file)
        for file in all_files
        if os.path.splitext(file)[1].lower() in image_extensions
    ]

    # Limit the number of images to process
    images_to_process = all_image_paths[:process_limit]
    
    if not images_to_process:
        print(f"No images found in the directory: {image_directory}")
    else:
        print(f"Processing {len(images_to_process)} images from the directory.")

        # Example usage of the new batched function
        results_batch = watermark_detector_batch(
            image_paths=images_to_process,
            run_yolo=False, 
            yolo_conf_thresh=0.7,
            owl_conf_thresh=0.8
        )

        if results_batch:
            print("\n--- Final Results ---")
            for res in results_batch:
                print(f"Image Path: {res['image_path']}")
                print(f"Is Watermarked: {res['is_watermarked']}")
                print(f"YOLO Bounding Boxes: {res['yolo_boxes']}")
                print("---")

Using device: cuda
Processing 10 images from the directory.


C:\Users\User\AppData\Local\Temp\ipykernel_30516\4107084658.py:77: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  owl_model.load_state_dict(torch.load("far5y1y5-8000.pt", map

OWLv2 model loaded.
YOLO model loaded.
Total processing time for batch of 10 images: 30.08 seconds

--- Final Results ---
Image Path: C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/downloaded_images/0000\1.jpg
Is Watermarked: False
YOLO Bounding Boxes: []
---
Image Path: C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/downloaded_images/0000\10.jpg
Is Watermarked: False
YOLO Bounding Boxes: []
---
Image Path: C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/downloaded_images/0000\100.jpg
Is Watermarked: False
YOLO Bounding Boxes: []
---
Image Path: C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/downloaded_images/0000\1000.jpg
Is Watermarked: False
YOLO Bounding Boxes: []
---
Image Path: C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/downloaded_

In [5]:
import os
import time
from PIL import Image
from ultralytics import YOLO
import torchvision.transforms.functional as TVF
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn.functional as F
from transformers import Owlv2VisionModel
from torch import nn

# --- Device Setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# --- OWLv2 Classification Head ---
class DetectorModelOwl(nn.Module):
    owl: Owlv2VisionModel

    def __init__(self, model_path: str, dropout: float, n_hidden: int = 768):
        super().__init__()

        owl = Owlv2VisionModel.from_pretrained(model_path).to(device)
        assert isinstance(owl, Owlv2VisionModel)
        self.owl = owl
        self.owl.requires_grad_(False)

        self.dropout1 = nn.Dropout(dropout)
        self.ln1 = nn.LayerNorm(n_hidden, eps=1e-5)
        self.linear1 = nn.Linear(n_hidden, n_hidden * 2)
        self.act1 = nn.GELU()
        self.dropout2 = nn.Dropout(dropout)
        self.ln2 = nn.LayerNorm(n_hidden * 2, eps=1e-5)
        self.linear2 = nn.Linear(n_hidden * 2, 2)

    def forward(self, pixel_values: torch.Tensor, labels: torch.Tensor | None = None):
        with torch.autocast(device.type, dtype=torch.bfloat16):
            outputs = self.owl(pixel_values=pixel_values, output_hidden_states=True)
            x = outputs.last_hidden_state  # B, N, C

            x = self.dropout1(x)
            x = self.ln1(x)
            x = self.linear1(x)
            x = self.act1(x)

            x = self.dropout2(x)
            x, _ = x.max(dim=1)
            x = self.ln2(x)

            x = self.linear2(x)

        if labels is not None:
            labels = labels.to(device)
            loss = F.cross_entropy(x, labels)
            return (x, loss)

        return (x,)


# --- Preprocessing function ---
def preprocess_image_for_owl_and_yolo(image_path: str):
    try:
        image = Image.open(image_path).convert("RGB")

        # Keep a copy for YOLO
        yolo_ready = image.copy()

        # OWLv2 preprocessing
        big_side = max(image.size)
        new_image = Image.new("RGB", (big_side, big_side), (128, 128, 128))
        new_image.paste(image, (0, 0))
        preped = new_image.resize((960, 960), Image.BICUBIC)

        preped = TVF.pil_to_tensor(preped) / 255.0
        owl_ready = TVF.normalize(
            preped,
            [0.48145466, 0.4578275, 0.40821073],
            [0.26862954, 0.26130258, 0.27577711]
        )

        return owl_ready, yolo_ready, image_path
    except Exception as e:
        print(f"Error loading {image_path}: {e}")
        return None, None, image_path


# --- Dataset ---
class ImageDataset(Dataset):
    def __init__(self, image_paths):
        self.image_paths = image_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        try:
            image = Image.open(path).convert("RGB")

            # OWLv2 preprocessing
            big_side = max(image.size)
            new_image = Image.new("RGB", (big_side, big_side), (128, 128, 128))
            new_image.paste(image, (0, 0))
            preped = new_image.resize((960, 960), Image.BICUBIC)

            preped = TVF.pil_to_tensor(preped) / 255.0
            owl_ready = TVF.normalize(
                preped,
                [0.48145466, 0.4578275, 0.40821073],
                [0.26862954, 0.26130258, 0.27577711]
            )

            return owl_ready, path
        except Exception as e:
            print(f"Error loading {path}: {e}")
            return None, path



# --- Batch Detector ---
def watermark_detector_batch(
    image_paths: list[str],
    run_yolo: bool,
    yolo_conf_thresh: float,
    owl_conf_thresh: float,
    batch_size: int = 8,
    num_workers: int = 4,
):
    global owl_model, yolo_model
    if owl_model is None:
        owl_model = DetectorModelOwl("google/owlv2-base-patch16-ensemble", dropout=0.0).to(device)
        owl_model.load_state_dict(torch.load("far5y1y5-8000.pt", map_location=device))
        owl_model.eval()
        print("OWLv2 model loaded.")
    if yolo_model is None:
        yolo_model = YOLO("yolo11x-train28-best.pt").to(device)
        print("YOLO model loaded.")

    dataset = ImageDataset(image_paths)
    loader = DataLoader(dataset, batch_size=batch_size, num_workers=num_workers, pin_memory=True)

    results = []
    start_time = time.time()

    for owl_inputs, paths in loader:
        # Filter out bad images
        valid_idx = [i for i, x in enumerate(owl_inputs) if x is not None]
        if not valid_idx:
            continue

        owl_batch = torch.stack([owl_inputs[i] for i in valid_idx]).to(device)

        # OWLv2 forward
        with torch.no_grad():
            logits, = owl_model(owl_batch, None)
        probs = F.softmax(logits, dim=1)

        # Collect results
        for i, idx in enumerate(valid_idx):
            watermark_prob = probs[i][1].item()
            is_watermarked = watermark_prob >= owl_conf_thresh
            results.append({
                "image_path": paths[idx],
                "is_watermarked": is_watermarked,
                "yolo_boxes": []
            })

        # YOLO part (reload images in main process)
        if run_yolo:
            yolo_images = [Image.open(paths[i]).convert("RGB") for i in valid_idx]
            yolo_results = yolo_model(yolo_images, imgsz=1024, augment=True,
                                    iou=0.5, conf=yolo_conf_thresh)

            for j, res in enumerate(yolo_results):
                yolo_boxes = []
                for box in res.boxes:
                    coords = box.xyxy[0].tolist()
                    yolo_boxes.append({
                        "class_id": int(box.cls.item()),
                        "confidence": float(box.conf.item()),
                        "bbox_coords": [round(c, 2) for c in coords]
                    })
                results[j]["yolo_boxes"] = yolo_boxes


    print(f"Processed {len(results)} images in {time.time()-start_time:.2f} sec")
    return results


# --- Globals ---
owl_model = None
yolo_model = None


# --- Main ---
if __name__ == "__main__":
    image_directory = "C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/downloaded_images/0000"
    process_limit = 10

    all_files = os.listdir(image_directory)
    image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp']
    all_image_paths = [
        os.path.join(image_directory, f)
        for f in all_files
        if os.path.splitext(f)[1].lower() in image_extensions
    ][:process_limit]

    if not all_image_paths:
        print(f"No images found in {image_directory}")
    else:
        results = watermark_detector_batch(
            image_paths=all_image_paths,
            run_yolo=False,
            yolo_conf_thresh=0.7,
            owl_conf_thresh=0.8,
            batch_size=4,
            num_workers=0,
        )
        print("\n--- Results ---")
        for r in results:
            print(r)


Using device: cuda


C:\Users\User\AppData\Local\Temp\ipykernel_30516\1742318266.py:133: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  owl_model.load_state_dict(torch.load("far5y1y5-8000.pt", ma

OWLv2 model loaded.
YOLO model loaded.
Processed 10 images in 23.63 sec

--- Results ---
{'image_path': 'C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/downloaded_images/0000\\1.jpg', 'is_watermarked': False, 'yolo_boxes': []}
{'image_path': 'C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/downloaded_images/0000\\10.jpg', 'is_watermarked': False, 'yolo_boxes': []}
{'image_path': 'C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/downloaded_images/0000\\100.jpg', 'is_watermarked': False, 'yolo_boxes': []}
{'image_path': 'C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/downloaded_images/0000\\1000.jpg', 'is_watermarked': False, 'yolo_boxes': []}
{'image_path': 'C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/downloaded_images/0000\\100001.jpg', 'is

In [ ]:
import os
from PIL import Image
from ultralytics import YOLO
import torchvision.transforms.functional as TVF
from transformers import Owlv2VisionModel
from torch import nn
import torch
import torch.nn.functional as F
import time
import cv2
import multiprocessing
import sys

# --- General Setup ---
# Check for GPU availability and define the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# OWLv2 classification head
class DetectorModelOwl(nn.Module):
    owl: Owlv2VisionModel

    def __init__(self, model_path: str, dropout: float, n_hidden: int = 768):
        super().__init__()
        owl = Owlv2VisionModel.from_pretrained(model_path).to(device)
        assert isinstance(owl, Owlv2VisionModel)
        self.owl = owl
        self.owl.requires_grad_(False)
        self.transforms = None

        self.dropout1 = nn.Dropout(dropout)
        self.ln1 = nn.LayerNorm(n_hidden, eps=1e-5)
        self.linear1 = nn.Linear(n_hidden, n_hidden * 2)
        self.act1 = nn.GELU()
        self.dropout2 = nn.Dropout(dropout)
        self.ln2 = nn.LayerNorm(n_hidden * 2, eps=1e-5)
        self.linear2 = nn.Linear(n_hidden * 2, 2)
    
    def forward(self, pixel_values: torch.Tensor, labels: torch.Tensor | None = None):
        with torch.autocast(device.type, dtype=torch.bfloat16):
            outputs = self.owl(pixel_values=pixel_values, output_hidden_states=True)
            x = outputs.last_hidden_state
            x = self.dropout1(x)
            x = self.ln1(x)
            x = self.linear1(x)
            x = self.act1(x)
            x = self.dropout2(x)
            x, _ = x.max(dim=1)
            x = self.ln2(x)
            x = self.linear2(x)
        
        if labels is not None:
            labels = labels.to(device)
            loss = F.cross_entropy(x, labels)
            return (x, loss)

        return (x,)

# Define the models as global variables
owl_model = None
yolo_model = None

def load_models():
    """Loads the pre-trained OWLv2 and YOLO models only once."""
    global owl_model, yolo_model
    try:
        if owl_model is None:
            owl_model = DetectorModelOwl("google/owlv2-base-patch16-ensemble", dropout=0.0).to(device)
            owl_model.load_state_dict(torch.load("far5y1y5-8000.pt", map_location=device))
            owl_model.eval()
            print("OWLv2 model loaded.")
        
        if yolo_model is None:
            yolo_model = YOLO("yolo11x-train28-best.pt").to(device)
            print("YOLO model loaded.")
        
        return True
    except Exception as e:
        print(f"Error loading models: {e}")
        return False

# Worker function for a single image's preprocessing
def preprocess_single_image(image_path):
    """Loads and preprocesses a single image, designed for multiprocessing."""
    try:
        image = Image.open(image_path).convert("RGB")
        
        # OWLv2 preprocessing (pad, resize, normalize)
        big_side = max(image.size)
        new_image = Image.new("RGB", (big_side, big_side), (128, 128, 128))
        new_image.paste(image, (0, 0))
        preped = new_image.resize((960, 960), Image.BICUBIC)
        
        preped = TVF.pil_to_tensor(preped) / 255.0
        input_image = TVF.normalize(preped, [0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711])

        return input_image, image, image_path
        
    except Exception as e:
        print(f"Error processing {image_path}: {e}", file=sys.stderr)
        return None, None, image_path

def watermark_detector_batch(
    image_paths: list[str], 
    run_yolo: bool, 
    yolo_conf_thresh: float, 
    owl_conf_thresh: float
):
    """
    Detects watermarks in a batch of images with parallelized preprocessing.
    """
    # Models are loaded in the main block before this function is called
    if owl_model is None or yolo_model is None:
        print("Models not loaded. Please call load_models() first.")
        return []

    start_time = time.time()
    
    # Use a multiprocessing pool to parallelize the CPU-bound preprocessing
    # The models are NOT reloaded in this step.
    with multiprocessing.Pool(processes=os.cpu_count()) as pool:
        preprocessed_results = pool.map(preprocess_single_image, image_paths)

    # # Filter out failed images and separate the tensors
    # valid_results = [res for res in preprocessed_results if res[0] is not None]
    # if not valid_results:
    #     print("No valid images found for processing.")
    #     return []

    # owl_inputs = [res[0] for res in valid_results]
    # yolo_image_list = [res[1] for res in valid_results]
    # processed_image_paths = [res[2] for res in valid_results]

    # # Concatenate into a single batch tensor for efficient GPU inference
    # owl_batch = torch.stack(owl_inputs).to(device)

    # # --- OWLv2 Batched Prediction (on GPU) ---
    # with torch.no_grad():
    #     logits, = owl_model(owl_batch, None)
    
    # probs = F.softmax(logits, dim=1)
    
    # results = []
    # for i, image_path in enumerate(processed_image_paths):
    #     watermark_prob = probs[i][1].item()
    #     is_watermarked = watermark_prob >= owl_conf_thresh
        
    #     results.append({
    #         "image_path": image_path,
    #         "is_watermarked": is_watermarked,
    #         "yolo_boxes": []
    #     })

    # # --- YOLO Batched Prediction (on GPU) ---
    # if run_yolo:
    #     yolo_results = yolo_model(yolo_image_list, imgsz=1024, augment=True, iou=0.5, conf=yolo_conf_thresh)
        
    #     for i, result in enumerate(yolo_results):
    #         yolo_boxes = []
    #         for box in result.boxes:
    #             coords = box.xyxy[0].tolist()
    #             yolo_boxes.append({
    #                 "class_id": int(box.cls.item()),
    #                 "confidence": float(box.conf.item()),
    #                 "bbox_coords": [round(c, 2) for c in coords]
    #             })
    #         results[i]["yolo_boxes"] = yolo_boxes

    # end_time = time.time()
    # total_time = end_time - start_time
    # print(f"Total processing time for batch of {len(processed_image_paths)} images: {total_time:.2f} seconds")

    # return results

    return None

# --- Main execution block ---
if __name__ == "__main__":
    # Ensure this script is the main entry point to prevent issues with multiprocessing on Windows
    # A single call to load_models() is all that's needed
    if not load_models():
        sys.exit("Model loading failed.")

    image_directory = 'C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/downloaded_images/0000/'
    process_limit = 10 
    
    all_files = os.listdir(image_directory)
    image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp']
    all_image_paths = [
        os.path.join(image_directory, file)
        for file in all_files
        if os.path.splitext(file)[1].lower() in image_extensions
    ]

    images_to_process = all_image_paths[:process_limit]
    
    if not images_to_process:
        print(f"No images found in the directory: {image_directory}")
    else:
        print(f"Processing {len(images_to_process)} images from the directory.")

        results_batch = watermark_detector_batch(
            image_paths=images_to_process,
            run_yolo=True, 
            yolo_conf_thresh=0.7,
            owl_conf_thresh=0.8
        )

        if results_batch:
            print("\n--- Final Results ---")
            for res in results_batch:
                print(f"Image Path: {res['image_path']}")
                print(f"Is Watermarked: {res['is_watermarked']}")
                print(f"YOLO Bounding Boxes: {res['yolo_boxes']}")
                print("---")

c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.conda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.conda\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\User\AppData\Local\Temp\ipykernel_8348\3144747100.py:69: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowl

OWLv2 model loaded.
YOLO model loaded.
Processing 10 images from the directory.


In [ ]:
import os
from PIL import Image
from ultralytics import YOLO
import torchvision.transforms.functional as TVF
from transformers import Owlv2VisionModel
from torch import nn
import torch
import torch.nn.functional as F
import time
import cv2
import multiprocessing
import sys

# --- General Setup ---
# Check for GPU availability and define the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# OWLv2 classification head
class DetectorModelOwl(nn.Module):
    owl: Owlv2VisionModel

    def __init__(self, model_path: str, dropout: float, n_hidden: int = 768):
        super().__init__()
        # Models are now loaded only in the main process
        owl = Owlv2VisionModel.from_pretrained(model_path).to(device)
        assert isinstance(owl, Owlv2VisionModel)
        self.owl = owl
        self.owl.requires_grad_(False)
        self.transforms = None

        self.dropout1 = nn.Dropout(dropout)
        self.ln1 = nn.LayerNorm(n_hidden, eps=1e-5)
        self.linear1 = nn.Linear(n_hidden, n_hidden * 2)
        self.act1 = nn.GELU()
        self.dropout2 = nn.Dropout(dropout)
        self.ln2 = nn.LayerNorm(n_hidden * 2, eps=1e-5)
        self.linear2 = nn.Linear(n_hidden * 2, 2)
    
    def forward(self, pixel_values: torch.Tensor, labels: torch.Tensor | None = None):
        with torch.autocast(device.type, dtype=torch.bfloat16):
            outputs = self.owl(pixel_values=pixel_values, output_hidden_states=True)
            x = outputs.last_hidden_state
            x = self.dropout1(x)
            x = self.ln1(x)
            x = self.linear1(x)
            x = self.act1(x)
            x = self.dropout2(x)
            x, _ = x.max(dim=1)
            x = self.ln2(x)
            x = self.linear2(x)
        
        if labels is not None:
            labels = labels.to(device)
            loss = F.cross_entropy(x, labels)
            return (x, loss)

        return (x,)

# Define the models as global variables
owl_model = None
yolo_model = None

def load_models():
    """Loads the pre-trained OWLv2 and YOLO models only once."""
    global owl_model, yolo_model
    try:
        if owl_model is None:
            owl_model = DetectorModelOwl("google/owlv2-base-patch16-ensemble", dropout=0.0).to(device)
            owl_model.load_state_dict(torch.load("far5y1y5-8000.pt", map_location=device))
            owl_model.eval()
            print("OWLv2 model loaded.")
        
        if yolo_model is None:
            yolo_model = YOLO("yolo11x-train28-best.pt").to(device)
            print("YOLO model loaded.")
        
        return True
    except Exception as e:
        print(f"Error loading models: {e}")
        return False

# Worker function for a single image's preprocessing
def preprocess_single_image(image_path):
    """Loads and preprocesses a single image, designed for multiprocessing."""
    try:
        # Use a context manager for the image to ensure it's closed properly
        with Image.open(image_path).convert("RGB") as image:
            # OWLv2 preprocessing (pad, resize, normalize)
            big_side = max(image.size)
            new_image = Image.new("RGB", (big_side, big_side), (128, 128, 128))
            new_image.paste(image, (0, 0))
            preped = new_image.resize((960, 960), Image.BICUBIC)
            
            preped = TVF.pil_to_tensor(preped) / 255.0
            input_image = TVF.normalize(preped, [0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711])

            # The multiprocessing pool will automatically handle pickling of these results
            return input_image, image, image_path
            
    except Exception as e:
        print(f"Error processing {image_path}: {e}", file=sys.stderr)
        return None, None, image_path

def run_detector_pipeline(
    image_paths: list[str], 
    run_yolo: bool, 
    yolo_conf_thresh: float, 
    owl_conf_thresh: float
):
    """
    Main function to run the entire detection pipeline.
    """
    start_time = time.time()
    
    # Use a multiprocessing pool to parallelize the CPU-bound preprocessing
    with multiprocessing.Pool(processes=os.cpu_count()) as pool:
        preprocessed_results = pool.map(preprocess_single_image, image_paths)

    # Filter out failed images and separate the tensors
    valid_results = [res for res in preprocessed_results if res[0] is not None]
    if not valid_results:
        print("No valid images found for processing.")
        return []

    owl_inputs = [res[0] for res in valid_results]
    yolo_image_list = [res[1] for res in valid_results]
    processed_image_paths = [res[2] for res in valid_results]

    # Concatenate into a single batch tensor for efficient GPU inference
    owl_batch = torch.stack(owl_inputs).to(device)

    # --- OWLv2 Batched Prediction (on GPU) ---
    with torch.no_grad():
        logits, = owl_model(owl_batch, None)
    
    probs = F.softmax(logits, dim=1)
    
    results = []
    for i, image_path in enumerate(processed_image_paths):
        watermark_prob = probs[i][1].item()
        is_watermarked = watermark_prob >= owl_conf_thresh
        
        results.append({
            "image_path": image_path,
            "is_watermarked": is_watermarked,
            "yolo_boxes": []
        })

    # --- YOLO Batched Prediction (on GPU) ---
    if run_yolo:
        yolo_results = yolo_model(yolo_image_list, imgsz=1024, augment=True, iou=0.5, conf=yolo_conf_thresh)
        
        for i, result in enumerate(yolo_results):
            yolo_boxes = []
            for box in result.boxes:
                coords = box.xyxy[0].tolist()
                yolo_boxes.append({
                    "class_id": int(box.cls.item()),
                    "confidence": float(box.conf.item()),
                    "bbox_coords": [round(c, 2) for c in coords]
                })
            results[i]["yolo_boxes"] = yolo_boxes

    end_time = time.time()
    total_time = end_time - start_time
    print(f"Total processing time for batch of {len(processed_image_paths)} images: {total_time:.2f} seconds")

    return results

# --- Main execution block ---
if __name__ == "__main__":
    # On Windows, using multiprocessing requires this protection
    if sys.platform.startswith('win'):
        multiprocessing.freeze_support()
    
    # Load models in the main process ONCE
    if not load_models():
        sys.exit("Model loading failed.")

    image_directory = 'C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/downloaded_images/0000/'
    process_limit = 10 
    
    all_files = os.listdir(image_directory)
    image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp']
    all_image_paths = [
        os.path.join(image_directory, file)
        for file in all_files
        if os.path.splitext(file)[1].lower() in image_extensions
    ]

    images_to_process = all_image_paths[:process_limit]
    
    if not images_to_process:
        print(f"No images found in the directory: {image_directory}")
    else:
        print(f"Processing {len(images_to_process)} images from the directory.")

        results_batch = run_detector_pipeline(
            image_paths=images_to_process,
            run_yolo=True, 
            yolo_conf_thresh=0.7,
            owl_conf_thresh=0.8
        )

        if results_batch:
            print("\n--- Final Results ---")
            for res in results_batch:
                print(f"Image Path: {res['image_path']}")
                print(f"Is Watermarked: {res['is_watermarked']}")
                print(f"YOLO Bounding Boxes: {res['yolo_boxes']}")
                print("---")

c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.conda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.conda\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\User\AppData\Local\Temp\ipykernel_31364\3362455286.py:70: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allow

OWLv2 model loaded.
YOLO model loaded.
Processing 10 images from the directory.


In [ ]:
import os
from PIL import Image
import torchvision.transforms.functional as TVF
import multiprocessing
import sys
import time

# --- Worker function for a single image's preprocessing ---
def preprocess_single_image(image_path):
    """
    Loads and preprocesses a single image. This function is designed to be run
    in parallel by a multiprocessing pool.
    """
    try:
        print(f"Processing image: {image_path}")
        # Use a context manager for the image to ensure it's closed properly
        with Image.open(image_path).convert("RGB") as image:
            # Preprocessing steps (pad, resize, normalize)
            big_side = max(image.size)
            new_image = Image.new("RGB", (big_side, big_side), (128, 128, 128))
            new_image.paste(image, (0, 0))
            preped = new_image.resize((960, 960), Image.BICUBIC)
            
            preped = TVF.pil_to_tensor(preped) / 255.0
            input_image = TVF.normalize(preped, [0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711])
            
            # Return the preprocessed tensor and original PIL image
            return input_image, image, image_path
            
    except Exception as e:
        print(f"Error processing {image_path}: {e}", file=sys.stderr)
        return None, None, image_path

# --- Main execution block for parallel preprocessing ---
if __name__ == "__main__":
    # Ensure this script is the main entry point for multiprocessing
    if sys.platform.startswith('win'):
        multiprocessing.freeze_support()

    # Define the directory and the processing limit
    image_directory = 'C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/downloaded_images/0000/'
    process_limit = 10 
    
    # Get a list of all image paths
    all_files = os.listdir(image_directory)
    image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp']
    all_image_paths = [
        os.path.join(image_directory, file)
        for file in all_files
        if os.path.splitext(file)[1].lower() in image_extensions
    ]

    # Limit the number of images to process
    images_to_process = all_image_paths[:process_limit]
    
    if not images_to_process:
        print(f"No images found in the directory: {image_directory}")
    else:
        start_time = time.time()
        print(f"Processing {len(images_to_process)} images from the directory using {os.cpu_count()} processes.")
        
        # Use a multiprocessing pool to parallelize the CPU-bound preprocessing
        with multiprocessing.Pool(processes=1) as pool: #os.cpu_count()) as pool:
            preprocessed_results = pool.map(preprocess_single_image, images_to_process)
        
        end_time = time.time()
        total_time = end_time - start_time
        print(f"Preprocessing completed in: {total_time:.2f} seconds")

        # You can now access the results
        valid_results = [res for res in preprocessed_results if res[0] is not None]
        print(f"Successfully preprocessed {len(valid_results)} images.")

Processing 10 images from the directory using 16 processes.


In [ ]:
import os
from PIL import Image
import multiprocessing

def load_single_image(image_path):
    """Worker function to load one image and its path."""
    try:
        image = Image.open(image_path).convert("RGB")
        return image_path, image
    except Exception as e:
        print(f"Error loading image {image_path}: {e}")
        return image_path, None

def load_images_in_parallel(image_paths):
    """Loads a list of images into memory using multiprocessing."""
    images_dict = {}
    with multiprocessing.Pool(processes=os.cpu_count()) as pool:
        results = pool.map(load_single_image, image_paths)
    
    for path, image in results:
        if image:
            images_dict[path] = image
    return images_dict

# Define the directory and the processing limit
image_directory = 'C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/downloaded_images/0000/'
process_limit = 10 

# Get a list of all image paths
all_files = os.listdir(image_directory)
image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp']
all_image_paths = [
    os.path.join(image_directory, file)
    for file in all_files
    if os.path.splitext(file)[1].lower() in image_extensions
]

all_images_dict = load_images_in_parallel(all_image_paths[:10])

In [3]:
import pandas as pd
import numpy as np
import faiss
import torch
from tqdm import tqdm
from PIL import Image
from typing import List, Optional
from pathlib import Path
import pyarrow.dataset as ds
import gc
from transformers import AutoProcessor, AutoModel

def find_similar_images_faiss_gpu_auto_batch(
    parquet_files: List[Path],
    model_name: str,
    text_prompt: Optional[str] = None,
    image_path: Optional[str] = None,
    top_n: int = 10,
    alpha: float = 0.5,
    similarity_threshold: float = 0.7,
    cpu_batch_size: int = 10000,
    max_gpu_memory_gb: float = 16.0  # maximum GPU memory to use for embeddings
) -> List[dict]:
    """
    FAISS GPU search with automatic batching if the dataset does not fit in GPU memory.
    - Splits dataset into GPU-sized chunks
    - Searches each chunk individually
    - Merges top-N results from all chunks
    """
    assert text_prompt or image_path, "Provide either text prompt or image path."

    device = "cuda" if torch.cuda.is_available() else "cpu"
    processor = AutoProcessor.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    # -------------------------------
    # Step 1: Compute query embedding
    # -------------------------------
    text_emb = None
    image_emb = None

    if text_prompt:
        inputs = processor(text=text_prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            text_features = model.get_text_features(**inputs)
        text_emb = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

    if image_path:
        image = Image.open(image_path).convert("RGB")
        inputs = processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            image_features = model.get_image_features(**inputs)
        image_emb = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

    if text_emb is not None and image_emb is not None:
        query_emb = alpha * text_emb + (1 - alpha) * image_emb
    elif text_emb is not None:
        query_emb = text_emb
    else:
        query_emb = image_emb

    query_emb = query_emb.cpu().numpy().astype("float32")
    faiss.normalize_L2(query_emb)

    # -------------------------------
    # Step 2: Prepare dataset and determine embedding dimension
    # -------------------------------
    dataset = ds.dataset(parquet_files, format="parquet")
    dim = None
    for batch in dataset.to_batches(batch_size=1000, use_threads=True):
        df = batch.to_pandas()
        embeddings_list = [np.array(e, dtype=np.float32) for e in df["embeddings_result"] if e is not None]
        if len(embeddings_list) > 0:
            dim = embeddings_list[0].shape[0]
            break
    if dim is None:
        raise ValueError("No valid embeddings found in the dataset!")

    # Estimate max embeddings per GPU chunk
    bytes_per_vector = dim * 4  # float32
    max_vectors_per_gpu = int(max_gpu_memory_gb * 1024**3 / bytes_per_vector)

    print(f"Embedding dimension: {dim}, max vectors per GPU chunk: {max_vectors_per_gpu}")

    # -------------------------------
    # Step 3: Process dataset in CPU batches and chunk GPU indexes
    # -------------------------------
    all_results = []

    gpu_res = faiss.StandardGpuResources()
    image_info_buffer = []
    embeddings_buffer = []

    total_rows = dataset.count_rows()
    print(f"Total rows in dataset: {total_rows}")

    batch_iter = tqdm(dataset.to_batches(batch_size=cpu_batch_size, use_threads=True), desc="Processing batches", total=(total_rows + cpu_batch_size - 1) // cpu_batch_size)
    for batch in batch_iter:
        df = batch.to_pandas()
        # parse embeddings
        valid_mask = [e is not None for e in df["embeddings_result"]]
        if sum(valid_mask) == 0:
            continue
        batch_embeddings = np.array([np.array(e, dtype=np.float32) for e in df["embeddings_result"] if e is not None])
        batch_info = df.loc[valid_mask, ["url", "caption", "original_image_index"]].reset_index(drop=True)

        embeddings_buffer.append(batch_embeddings)
        image_info_buffer.extend(batch_info.to_dict("records"))

        # If buffer exceeds max GPU vectors, flush to GPU and search
        if sum(e.shape[0] for e in embeddings_buffer) >= max_vectors_per_gpu:
            # Concatenate buffered embeddings
            concat_emb = np.vstack(embeddings_buffer)
            faiss.normalize_L2(concat_emb)

            # Build GPU index
            cpu_index = faiss.IndexFlatIP(dim)
            gpu_index = faiss.index_cpu_to_gpu(gpu_res, 0, cpu_index)
            gpu_index.add(concat_emb)

            # Search query
            D, I = gpu_index.search(query_emb, top_n * 2)
            for dist, idx in zip(D[0], I[0]):
                if dist >= similarity_threshold:
                    info = image_info_buffer[idx]
                    all_results.append({
                        "url": info["url"],
                        "caption": info["caption"],
                        "original_image_index": info["original_image_index"],
                        "cosine_similarity": float(dist)
                    })

            # Clear buffers
            embeddings_buffer = []
            image_info_buffer = []
            del cpu_index, gpu_index, concat_emb
            gc.collect()

    # Process remaining embeddings if any
    if embeddings_buffer:
        concat_emb = np.vstack(embeddings_buffer)
        faiss.normalize_L2(concat_emb)
        cpu_index = faiss.IndexFlatIP(dim)
        gpu_index = faiss.index_cpu_to_gpu(gpu_res, 0, cpu_index)
        gpu_index.add(concat_emb)
        D, I = gpu_index.search(query_emb, top_n * 2)
        for dist, idx in zip(D[0], I[0]):
            if dist >= similarity_threshold:
                info = image_info_buffer[idx]
                all_results.append({
                    "url": info["url"],
                    "caption": info["caption"],
                    "original_image_index": info["original_image_index"],
                    "cosine_similarity": float(dist)
                })

        del cpu_index, gpu_index, concat_emb
        gc.collect()

    # Keep only top_n results across all chunks
    all_results = sorted(all_results, key=lambda x: x["cosine_similarity"], reverse=True)[:top_n]

    return all_results


In [1]:
import pandas as pd
import numpy as np
import faiss
import torch
from tqdm import tqdm
from PIL import Image
from typing import List, Optional
from pathlib import Path
import pyarrow.dataset as ds
import gc
from transformers import AutoProcessor, AutoModel

def find_similar_images_faiss_gpu_auto_batch(
    parquet_files: List[Path],
    model_name: str,
    text_prompt: Optional[str] = None,
    image_path: Optional[str] = None,
    top_n: Optional[int] = None,  # allow None to return all above threshold
    alpha: float = 0.5,
    similarity_threshold: float = 0.7,
    cpu_batch_size: int = 10000,
    max_gpu_memory_gb: float = 16.0
) -> List[dict]:
    """
    FAISS GPU search with automatic batching and threshold filtering.
    - Supports returning all images above similarity_threshold if top_n is None.
    - Automatically batches embeddings to fit in GPU memory.
    - Merges results across all chunks.
    """

    assert text_prompt or image_path, "Provide either text prompt or image path."

    device = "cuda" if torch.cuda.is_available() else "cpu"
    processor = AutoProcessor.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    # -------------------------------
    # Step 1: Compute query embedding
    # -------------------------------
    text_emb = None
    image_emb = None

    if text_prompt:
        inputs = processor(text=text_prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            text_features = model.get_text_features(**inputs)
        text_emb = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

    if image_path:
        image = Image.open(image_path).convert("RGB")
        inputs = processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            image_features = model.get_image_features(**inputs)
        image_emb = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

    if text_emb is not None and image_emb is not None:
        query_emb = alpha * text_emb + (1 - alpha) * image_emb
    elif text_emb is not None:
        query_emb = text_emb
    else:
        query_emb = image_emb

    query_emb = query_emb.cpu().numpy().astype("float32")
    faiss.normalize_L2(query_emb)

    # -------------------------------
    # Step 2: Prepare dataset and determine embedding dimension
    # -------------------------------
    dataset = ds.dataset(parquet_files, format="parquet")
    dim = None
    for batch in dataset.to_batches(batch_size=1000, use_threads=True):
        df = batch.to_pandas()
        embeddings_list = [np.array(e, dtype=np.float32) for e in df["embeddings_result"] if e is not None]
        if len(embeddings_list) > 0:
            dim = embeddings_list[0].shape[0]
            break
    if dim is None:
        raise ValueError("No valid embeddings found in the dataset!")

    # Estimate max embeddings per GPU chunk
    bytes_per_vector = dim * 4  # float32
    max_vectors_per_gpu = int(max_gpu_memory_gb * 1024**3 / bytes_per_vector)

    print(f"Embedding dimension: {dim}, max vectors per GPU chunk: {max_vectors_per_gpu}")

    # -------------------------------
    # Step 3: Process dataset in CPU batches and chunk GPU indexes
    # -------------------------------
    all_results = []

    gpu_res = faiss.StandardGpuResources()
    image_info_buffer = []
    embeddings_buffer = []

    total_rows = dataset.count_rows()
    batch_iter = tqdm(
        dataset.to_batches(batch_size=cpu_batch_size, use_threads=True),
        desc="Processing batches",
        total=(total_rows + cpu_batch_size - 1) // cpu_batch_size
    )

    for batch in batch_iter:
        df = batch.to_pandas()
        # parse embeddings
        valid_mask = [e is not None for e in df["embeddings_result"]]
        if sum(valid_mask) == 0:
            continue
        batch_embeddings = np.array([np.array(e, dtype=np.float32) for e in df["embeddings_result"] if e is not None])
        batch_info = df.loc[valid_mask, ["url", "caption", "original_image_index"]].reset_index(drop=True)

        # -------------------------------
        # Directly flush each CPU batch to GPU to avoid MemoryError
        # -------------------------------
        faiss.normalize_L2(batch_embeddings)
        cpu_index = faiss.IndexFlatIP(dim)
        gpu_index = faiss.index_cpu_to_gpu(gpu_res, 0, cpu_index)
        gpu_index.add(batch_embeddings)

        # Determine search k for GPU (max 2048)
        search_k = min(batch_embeddings.shape[0], 2048)
        D, I = gpu_index.search(query_emb, search_k)

        # Collect results above threshold
        for dist, idx in zip(D[0], I[0]):
            if dist >= similarity_threshold:
                info = batch_info.iloc[idx]
                all_results.append({
                    "url": info["url"],
                    "caption": info["caption"],
                    "original_image_index": info["original_image_index"],
                    "cosine_similarity": float(dist)
                })

        del batch_embeddings, batch_info, cpu_index, gpu_index, D, I
        gc.collect()

    # -------------------------------
    # Step 4: Merge and return results
    # -------------------------------
    # Sort by similarity descending
    all_results = sorted(all_results, key=lambda x: x["cosine_similarity"], reverse=True)

    # Apply top_n only if specified
    if top_n is not None:
        all_results = all_results[:top_n]

    return all_results


c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.conda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
import pandas as pd
import numpy as np
import faiss
import torch
from tqdm import tqdm
from PIL import Image
from typing import List, Optional
from pathlib import Path
import pyarrow.dataset as ds
import gc
from transformers import AutoProcessor, AutoModel
import ast

def parse_embedding(e):
    if e is None:
        return None
    try:
        if isinstance(e, str):
            e = ast.literal_eval(e)
        return np.array(e, dtype=np.float32)
    except Exception:
        return None

def find_similar_images_faiss_gpu_auto_batch(
    parquet_files: List[Path],
    model_name: str,
    text_prompt: Optional[str] = None,
    image_path: Optional[str] = None,
    top_n: Optional[int] = None,  # allow None to return all above threshold
    alpha: float = 0.5,
    similarity_threshold: float = 0.7,
    cpu_batch_size: int = 10000,
    max_gpu_memory_gb: float = 16.0
) -> List[dict]:
    """
    FAISS GPU search with automatic batching and threshold filtering.
    - Supports returning all images above similarity_threshold if top_n is None.
    - Automatically batches embeddings to fit in GPU memory.
    - Splits each CPU batch into sub-batches ≤ 2048 to respect FAISS GPU max-k limit.
    - Merges results across all chunks.
    """

    assert text_prompt or image_path, "Provide either text prompt or image path."

    device = "cuda" if torch.cuda.is_available() else "cpu"
    processor = AutoProcessor.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    # -------------------------------
    # Step 1: Compute query embedding
    # -------------------------------
    text_emb = None
    image_emb = None

    if text_prompt:
        inputs = processor(text=text_prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            text_features = model.get_text_features(**inputs)
        text_emb = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

    if image_path:
        image = Image.open(image_path).convert("RGB")
        inputs = processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            image_features = model.get_image_features(**inputs)
        image_emb = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

    if text_emb is not None and image_emb is not None:
        query_emb = alpha * text_emb + (1 - alpha) * image_emb
    elif text_emb is not None:
        query_emb = text_emb
    else:
        query_emb = image_emb

    query_emb = query_emb.cpu().numpy().astype("float32")
    faiss.normalize_L2(query_emb)

    # -------------------------------
    # Step 2: Prepare dataset and determine embedding dimension
    # -------------------------------
    dataset = ds.dataset(parquet_files, format="parquet")
    dim = None
    for batch in dataset.to_batches(batch_size=1000, use_threads=True):
        df = batch.to_pandas()
        embeddings_list = [parse_embedding(e) for e in df["embeddings_result"] if e is not None]
        if len(embeddings_list) > 0:
            dim = embeddings_list[0].shape[0]
            break
    if dim is None:
        raise ValueError("No valid embeddings found in the dataset!")

    # Estimate max embeddings per GPU chunk
    bytes_per_vector = dim * 4  # float32
    max_vectors_per_gpu = int(max_gpu_memory_gb * 1024**3 / bytes_per_vector)
    print(f"Embedding dimension: {dim}, max vectors per GPU chunk: {max_vectors_per_gpu}")

    # -------------------------------
    # Step 3: Process dataset in CPU batches and chunk GPU indexes
    # -------------------------------
    all_results = []

    gpu_res = faiss.StandardGpuResources()
    total_rows = dataset.count_rows()
    batch_iter = tqdm(
        dataset.to_batches(batch_size=cpu_batch_size, use_threads=True),
        desc="Processing CPU batches",
        total=(total_rows + cpu_batch_size - 1) // cpu_batch_size
    )

    for batch in batch_iter:
        df = batch.to_pandas()
        # parse embeddings
        valid_mask = [e is not None for e in df["embeddings_result"]]
        if sum(valid_mask) == 0:
            continue
        batch_embeddings = np.array([parse_embedding(e) for e in df["embeddings_result"] if e is not None])
        batch_info = df.loc[valid_mask, ["url", "caption", "original_image_index"]].reset_index(drop=True)

        # Normalize embeddings
        faiss.normalize_L2(batch_embeddings)

        # -------------------------------
        # Split CPU batch into sub-batches ≤ 2048 for GPU
        # -------------------------------
        sub_batch_size = 2048
        num_vectors = batch_embeddings.shape[0]
        for start_idx in range(0, num_vectors, sub_batch_size):
            end_idx = min(start_idx + sub_batch_size, num_vectors)
            sub_embeddings = batch_embeddings[start_idx:end_idx]
            sub_info = batch_info.iloc[start_idx:end_idx]

            # Build GPU index for this sub-batch
            cpu_index = faiss.IndexFlatIP(dim)
            gpu_index = faiss.index_cpu_to_gpu(gpu_res, 0, cpu_index)
            gpu_index.add(sub_embeddings)

            # Search all vectors in this sub-batch
            D, I = gpu_index.search(query_emb, sub_embeddings.shape[0])  # safe: ≤2048

            # Collect results above threshold
            for dist, idx in zip(D[0], I[0]):
                if dist >= similarity_threshold:
                    info = sub_info.iloc[idx]
                    all_results.append({
                        "url": info["url"],
                        "caption": info["caption"],
                        "original_image_index": info["original_image_index"],
                        "cosine_similarity": float(dist)
                    })

            del sub_embeddings, sub_info, cpu_index, gpu_index, D, I
            gc.collect()

    # -------------------------------
    # Step 4: Merge and return results
    # -------------------------------
    # Sort by similarity descending
    all_results = sorted(all_results, key=lambda x: x["cosine_similarity"], reverse=True)

    # Apply top_n only if specified
    if top_n is not None:
        all_results = all_results[:top_n]

    return all_results


In [18]:
import os
dataset_dir = r'C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\clip_embeddings_resumable_symlink\all_images_openai_clip_vit_large_patch14\0000_embeddings'

all_files = os.listdir(dataset_dir)
image_extensions = ['.parquet']
all_parquet_file_paths = [
    os.path.join(dataset_dir, f)
    for f in all_files
    if os.path.splitext(f)[1].lower() in image_extensions
]

# print(f"Found {len(all_parquet_file_paths)} parquet files in {dataset_dir}")

clip_model_names = ["openai/clip-vit-base-patch16", "openai/clip-vit-base-patch32", "openai/clip-vit-large-patch14"]
model_name = clip_model_names[2] 
text_prompt = "watermark" #"female construction worker"
search_image_path = r'C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\images\test\TestConstructionWorkerImage.jpg' 

image_info_list_faiss = find_similar_images_faiss_gpu_auto_batch(
    parquet_files=all_parquet_file_paths,
    model_name=model_name,
    text_prompt=text_prompt,
    image_path=search_image_path,
    top_n=None,
    alpha=1, # 1 = text only, 0 = image only, 0.5 = equal weighting
    similarity_threshold=0.15,
    cpu_batch_size = 10000,
    max_gpu_memory_gb = 7.5
)

c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.conda\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Embedding dimension: 768, max vectors per GPU chunk: 2621440


Processing CPU batches: 100%|██████████| 1639/1639 [35:18<00:00,  1.29s/it] 


In [6]:
len(image_info_list)

4290547

In [4]:
# Sort the list in ascending order of original_image_index
image_info_list_sorted = sorted(image_info_list, key=lambda x: x['original_image_index'])

# # Verify by printing all entries with original_image_index == 3
# for x in image_info_list_sorted:
#     if x['original_image_index'] == 104:
#         print(x)

# image_info_list_sorted[:10]
# image_info_list[-10:]

In [5]:
image_info_list_sorted[:10]

[{'url': 'https://c8.alamy.com/comp/C84HAM/herd-of-cows-on-alpine-pasture-among-mountains-in-alps-northern-italy-C84HAM.jpg',
  'caption': 'Herd of cows on alpine pasture among mountains in Alps, northern Italy. Stock Photo',
  'original_image_index': 3,
  'cosine_similarity': 0.2062414288520813},
 {'url': 'https://blog.spoongraphics.co.uk/wp-content/uploads/2008/01/iron_man_ver2.jpg',
  'caption': 'Iron Man Movie Poster',
  'original_image_index': 6,
  'cosine_similarity': 0.17893677949905396},
 {'url': 'https://thumb1.shutterstock.com/image-photo/redirected-150nw-342752588.jpg',
  'caption': 'happy father reading book with... | Shutterstock . vector #342752588',
  'original_image_index': 8,
  'cosine_similarity': 0.168187215924263},
 {'url': 'http://a4.pbase.com/t6/50/322150/4/72331488.VWZjtoEZ.jpg',
  'caption': 'Thirsty sparrow',
  'original_image_index': 12,
  'cosine_similarity': 0.1746973991394043},
 {'url': 'https://image.spreadshirtmedia.com/image-server/v1/mp/products/T499A2P

In [3]:
def find_similar_images(
    dataset_dir: str,
    model_name: str,
    text_prompt: Optional[str] = None,
    image_path: Optional[str] = None,
    top_n: Optional[int] = None,
    similarity_threshold: Optional[float] = None,
    alpha: float = 0.5
) -> List[dict]:
    ''' 
    Functions used to find images similar to a text or image (or both) based on the CLIP embeddings.

    Inputs:
    - dataset_dir: Directory containing the CLIP embeddings dataset in Parquet format.
    - model_name: Name of the pre-trained CLIP model to use.
    - text_prompt: Optional text prompt to find similar images. (None means only the image is used to search).
    - image_path: Optional path to an image to find similar images. (None means only the text prompt is used to search).
    - top_n: Optional number of top similar images to return. (None means return all).
    - similarity_threshold: Optional threshold for cosine similarity to filter results. (None means no filtering).
    - alpha: Weight for text similarity in the combined similarity score. (0.5 - equal weight between text and image, 1 - text prompt only is used, 0 - image only is used).

    Outputs:
    - List of dictionaries containing URLs, captions, original image indices, and cosine similarities of the most similar images.
    '''

    assert text_prompt or image_path, "You must provide either a text prompt or an image path."

    processor = AutoProcessor.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    text_embedding = None
    image_embedding = None

    if text_prompt:
        inputs = processor(text=text_prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            text_features = model.get_text_features(**inputs)
        text_embedding = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

    if image_path:
        image = Image.open(image_path).convert("RGB")
        inputs = processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            image_features = model.get_image_features(**inputs)
        image_embedding = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

    # Convert to numpy for similarity calculation
    text_embedding_np = text_embedding.cpu().squeeze().numpy() if text_embedding is not None else None
    image_embedding_np = image_embedding.cpu().squeeze().numpy() if image_embedding is not None else None

    all_matches = []

    # Loads only matching files from dataset_dir like part-00000.parquet to avoid loading checkpoints
    pattern = re.compile(r"^part-\d{5}\.parquet$")
    valid_files = [
        os.path.join(dataset_dir, f)
        for f in os.listdir(dataset_dir)
        if pattern.match(f)
    ]
    dataset = ds.dataset(valid_files, format="parquet")

    # Iterate through each fragment (.parquet file) composing the dataset
    for fragment in dataset.get_fragments():
        print(f"Processing fragment: {fragment.path}")
        pq_file = pq.ParquetFile(os.path.join(dataset_dir, fragment.path))

        for row_group_index in range(pq_file.num_row_groups):
            table = pq_file.read_row_group(row_group_index)
            df_chunk = table.to_pandas()

            df_chunk = df_chunk.dropna(subset=['embeddings_result'])
            if df_chunk.empty:
                continue

            df_chunk['embeddings_result'] = df_chunk['embeddings_result'].apply(
                lambda x: np.array(x) if isinstance(x, list) else x
            )
            image_embeddings = np.vstack(df_chunk['embeddings_result'].values)

            # Compute similarities separately
            sim_text = np.dot(image_embeddings, text_embedding_np.T) if text_embedding_np is not None else 0
            sim_image = np.dot(image_embeddings, image_embedding_np.T) if image_embedding_np is not None else 0

            # Combine similarity with weights
            combined_sim = None
            if text_embedding_np is not None and image_embedding_np is not None:
                combined_sim = alpha * sim_text + (1 - alpha) * sim_image
            elif text_embedding_np is not None:
                combined_sim = sim_text
            else:
                combined_sim = sim_image

            df_chunk['combined_similarity'] = combined_sim

            if similarity_threshold is not None:
                df_chunk = df_chunk[df_chunk['combined_similarity'] >= similarity_threshold]

            for _, row in df_chunk.iterrows():
                all_matches.append({
                    'url': row.get('url'),
                    'caption': row.get('caption'),
                    'original_image_index': row.get('original_image_index'),
                    'cosine_similarity': row['combined_similarity']
                })

    all_matches.sort(key=lambda x: x['cosine_similarity'], reverse=True)
    top_matches = all_matches[:top_n] if top_n is not None else all_matches
    # top_matches.reverse()

    return top_matches #all_matches[:top_n] if top_n is not None else all_matches

In [15]:
import os
import re
# import sys
# import glob
# import time
import torch
# import requests
import numpy as np
# import pandas as pd
# import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
from PIL import Image
# from io import BytesIO
# from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoModel
# from concurrent.futures import ThreadPoolExecutor
from typing import Optional, List
# from IPython.display import display

In [16]:
dataset_dir = r'C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\clip_embeddings_resumable_symlink\all_images_openai_clip_vit_large_patch14\0000_embeddings'
clip_model_names = ["openai/clip-vit-base-patch16", "openai/clip-vit-base-patch32", "openai/clip-vit-large-patch14"]
model_name = clip_model_names[2] 
text_prompt = "watermark"
search_image_path = r'C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\images\test\TestConstructionWorkerImage.jpg' 
top_n = None#10
similarity_threshold = 0.15

image_info_list_not_faiss = find_similar_images(
    dataset_dir=dataset_dir,
    model_name=model_name,
    text_prompt=text_prompt,
    image_path = search_image_path,
    top_n=top_n,
    similarity_threshold=similarity_threshold,
    alpha=1  # 1 = text only, 0 = image only, 0.5 = equal weighting
)

# TO-DO: Improve the execution time of the find_similar_images function.

c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.conda\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Processing fragment: C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/all_images_openai_clip_vit_large_patch14/0000_embeddings/part-00000.parquet
Processing fragment: C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/all_images_openai_clip_vit_large_patch14/0000_embeddings/part-00001.parquet
Processing fragment: C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/all_images_openai_clip_vit_large_patch14/0000_embeddings/part-00002.parquet
Processing fragment: C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/all_images_openai_clip_vit_large_patch14/0000_embeddings/part-00003.parquet
Processing fragment: C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/all_images_openai_clip_vit_large_patch14/0000_embeddings/part-00004.parquet
Processing fragment: C:/Master